# Week 2 — The model is just a rule you can read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cokezero20/FlyRank_AI_ML_Internship_NATIVIDAD/blob/main/notebooks/02_your_first_readable_model.ipynb?flush_cache=true)

You'll:
1. Write a **1-line hand rule** and rank pages with it.
2. Fit a **depth-2 decision tree** and `print` it — the model "learned" a readable if/else. Then compare — where does it beat your rule, and where doesn’t it?
3. See **why you never feed the outcome back in** — that's leakage.

The payoff isn't a high score. It's: *my intuition was rough, the model found the real signal, and I can read exactly what it found.*

## 0. Setup (Colab or local)

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

30000 pages |  declining rate: 0.542


## 1. A rule you write by hand: `stale x visible`
Intuition: a page worth reviewing is one that is **stale** (not updated in a while) **and** still **visible** (getting impressions). Rank those by how much exposure they have.

In [2]:
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]]

,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
16751,61678,194,19.7,0.15,down
16514,59472,194,24.8,0.13,down
7021,25715,194,22.2,0.23,down
21268,13299,193,10.5,0.49,down
11489,7812,194,39.0,0.01,down
12045,7558,193,17.9,0.20,down
698,4590,194,31.0,0.00,down
5327,4556,194,16.4,0.33,down
26810,4429,194,25.3,0.38,down
20837,1697,193,15.8,0.12,down


We need a way to score any ranking. **Precision@K** = of the top K pages a ranking flags, what fraction are actually declining.

In [3]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")

Hand rule  Precision@20: 0.900
Hand rule  Precision@50: 0.680


## 2. Let a model learn the rule — then read it
A **depth-2 decision tree** can only ask 3 yes/no questions. That constraint is the point: whatever it learns, you can read.

We give it a few **pre-decision** signals — never product flags.

In [4]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0



That printout **is** the model — a human-readable if/else. Now rank pages by the tree's probability and score it the same way.

In [5]:
tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

Precision@20:  hand rule 0.900   vs   tree 0.550
Precision@50:  hand rule 0.680   vs   tree 0.600


Look closely: the tree **wins at Precision@50** but your hand rule **wins at Precision@20**. Both results are real. A sharp human rule can be excellent at the very top of the list; the model's advantage shows up deeper, where simple rules run out of signal. Saying exactly that — instead of "the model is better" — is what honest evaluation sounds like.

## 3. Why you can't feed the outcome back in
Your label is `trend_direction == "down"`, and `trend_pct` is the exact percentage change that bucket is computed from — so it **is** the answer in disguise. Watch what happens if you feed it in as a feature:

In [6]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))

'Leaky' tree Precision@50: 1.000  <- looks amazing
|--- trend_pct <= -20.05
|   |--- word_count <= 212.00
|   |   |--- class: 1
|   |--- word_count >  212.00
|   |   |--- class: 1
|--- trend_pct >  -20.05
|   |--- trend_pct <= -19.95
|   |   |--- class: 0
|   |--- trend_pct >  -19.95
|   |   |--- class: 0



The tree just split on `trend_pct` and nailed the label — because the label is **derived from** `trend_pct`. That's **leakage**: the feature is the answer in disguise, and it teaches you nothing.

That's also why the starter data ships **only observable signals** — the product's own decision flags (health scores, "needs CTR fix", and so on) aren't included, so you can't accidentally train on them. You build from what was knowable *before* the outcome.

> Rule of thumb: if a feature would only be known *because someone already made the decision you're predicting*, it leaks. Leave it out.

## 4. 🔧 Your turn
- Change `max_depth` to 3 or 4 — does Precision@50 improve? Can you still read the tree?
- Swap in different features (drop `impressions_90d`, add `engagement_rate`). What does the tree choose to split on first?
- **Important caveat:** we scored *in-sample* here for teaching. The real pipeline uses **client-holdout** validation (`scripts/03_train_model.py`) so a client's pages never appear in both train and test. Re-run your comparison with a train/test split and see if the gap holds.

Write your experiment below.

In [ ]:
# Your experiment here


### Experiment 1: `max_depth = 3`

In [9]:
print('--- Experimenting with max_depth = 3 ---')
tree_depth3 = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
tree_depth3.fit(X, y)

tree_score_depth3 = tree_depth3.predict_proba(X)[:, 1]
hr_k50 = precision_at_k(df["hand_rule_score"], y, 50)
tr_k50_depth3 = precision_at_k(tree_score_depth3, y, 50)
print(f"Precision@50:  hand rule {hr_k50:.3f}   vs   tree (depth=3) {tr_k50_depth3:.3f}")

print('\nTree structure (max_depth=3):')
print(export_text(tree_depth3, feature_names=features))

--- Experimenting with max_depth = 3 ---
Precision@50:  hand rule 0.680   vs   tree (depth=3) 0.720

Tree structure (max_depth=3):
|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- impressions_90d <= 3.50
|   |   |   |--- class: 0
|   |   |--- impressions_90d >  3.50
|   |   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- content_age_days <= 108.50
|   |   |   |--- class: 0
|   |   |--- content_age_days >  108.50
|   |   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- ctr <= 0.33
|   |   |   |--- class: 1
|   |   |--- ctr >  0.33
|   |   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- avg_position <= 25.15
|   |   |   |--- class: 0
|   |   |--- avg_position >  25.15
|   |   |   |--- class: 0



###`max_depth = 4`

In [10]:
print('--- Experimenting with max_depth = 4 ---')
tree_depth4 = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)
tree_depth4.fit(X, y)

tree_score_depth4 = tree_depth4.predict_proba(X)[:, 1]
hr_k50 = precision_at_k(df["hand_rule_score"], y, 50)
tr_k50_depth4 = precision_at_k(tree_score_depth4, y, 50)
print(f"Precision@50:  hand rule {hr_k50:.3f}   vs   tree (depth=4) {tr_k50_depth4:.3f}")

print('\nTree structure (max_depth=4):')
print(export_text(tree_depth4, feature_names=features))

--- Experimenting with max_depth = 4 ---
Precision@50:  hand rule 0.680   vs   tree (depth=4) 0.680

Tree structure (max_depth=4):
|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- impressions_90d <= 3.50
|   |   |   |--- word_count <= 687.00
|   |   |   |   |--- class: 0
|   |   |   |--- word_count >  687.00
|   |   |   |   |--- class: 0
|   |   |--- impressions_90d >  3.50
|   |   |   |--- content_age_days <= 237.50
|   |   |   |   |--- class: 0
|   |   |   |--- content_age_days >  237.50
|   |   |   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- content_age_days <= 108.50
|   |   |   |--- days_since_last_update <= 14.00
|   |   |   |   |--- class: 1
|   |   |   |--- days_since_last_update >  14.00
|   |   |   |   |--- class: 0
|   |   |--- content_age_days >  108.50
|   |   |   |--- impressions_90d <= 2.50
|   |   |   |   |--- class: 0
|   |   |   |--- impressions_90d >  2.50
|   |   |   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- cont

max_depth=3 Tree: This tree was a good improvement! It gave a Precision@50 of 0.720, which is better than the original hand rule's 0.680. You found it readable, suggesting it struck a good balance between finding a better signal and remaining understandable.

max_depth=4 Tree: This tree became more complex, adding another layer of decisions. However, its Precision@50 actually dropped back down to 0.680, which is the same as the hand rule and worse than the max_depth=3 tree. While you still found it readable, its increased complexity didn't lead to better performance for Precision@50; instead, it seems to have captured more detail that didn't help, and might even have started to overcomplicate the rules for the top predictions.

In short: The max_depth=3 tree was the sweet spot, providing a performance boost while remaining clear. The max_depth=4 tree got more complex without improving (and even decreasing) the Precision@50 score, suggesting that the extra depth wasn't beneficial for this specific metric.

### Experiment 2: Swap features (drop `impressions_90d`, add `engagement_rate`)

In [14]:
print('--- Experimenting with new features and max_depth = 2 ---')

# Define the new set of features
new_features_depth2 = [
    "content_age_days",
    "days_since_last_update",
    "engagement_rate", # Added new feature
    "avg_position",
    "ctr",
    "word_count"
]

# Prepare the data with the new features, handling infinities and NaNs
X_new_features_depth2 = df[new_features_depth2].replace([np.inf, -np.inf], np.nan).fillna(0)

# Train a Decision Tree with max_depth=2
tree_new_features_depth2 = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree_new_features_depth2.fit(X_new_features_depth2, y)

# Calculate Precision@K for the new tree
tree_score_new_features_depth2 = tree_new_features_depth2.predict_proba(X_new_features_depth2)[:, 1]

hr_k20 = precision_at_k(df["hand_rule_score"], y, 20)
tr_k20_new_features_depth2 = precision_at_k(tree_score_new_features_depth2, y, 20)
print(f"Precision@20:  hand rule {hr_k20:.3f}   vs   tree (new features, depth=2) {tr_k20_new_features_depth2:.3f}")

hr_k50 = precision_at_k(df["hand_rule_score"], y, 50)
tr_k50_new_features_depth2 = precision_at_k(tree_score_new_features_depth2, y, 50)
print(f"Precision@50:  hand rule {hr_k50:.3f}   vs   tree (new features, depth=2) {tr_k50_new_features_depth2:.3f}")

print('\nTree structure (new features, max_depth=2):')
print(export_text(tree_new_features_depth2, feature_names=new_features_depth2))

--- Experimenting with new features and max_depth = 2 ---
Precision@20:  hand rule 0.900   vs   tree (new features, depth=2) 0.850
Precision@50:  hand rule 0.680   vs   tree (new features, depth=2) 0.700

Tree structure (new features, max_depth=2):
|--- avg_position <= 0.55
|   |--- avg_position <= 0.15
|   |   |--- class: 0
|   |--- avg_position >  0.15
|   |   |--- class: 0
|--- avg_position >  0.55
|   |--- content_age_days <= 287.50
|   |   |--- class: 1
|   |--- content_age_days >  287.50
|   |   |--- class: 0



*   Precision@20: The hand rule scored 0.900, while our new tree scored 0.850. This means for the very top 20 pages, your hand rule was a bit better at finding declining pages.

*   Precision@50: The hand rule scored 0.680, and our new tree scored 0.700. This shows that when we look at the top 50 pages, the tree with the new features does slightly better than your hand rule.



Now, for what the tree decided was most important to split on first: it's avg_position. The very first question the tree asks is: 'Is the avg_position less than or equal to 0.55?' This tells us that avg_position is the most significant factor for the tree right at the start when trying to identify declining pages with these new features and a shallow depth.

### Implementing Client-Holdout Validation

In [15]:
from sklearn.model_selection import train_test_split

# 1. Get unique client IDs
all_client_ids = df['client_id'].unique()

# 2. Split client IDs into train and test sets (80/20 split)
train_client_ids, test_client_ids = train_test_split(all_client_ids, test_size=0.2, random_state=42)

# 3. Create train and test DataFrames based on client IDs
train_df = df[df['client_id'].isin(train_client_ids)].copy()
test_df = df[df['client_id'].isin(test_client_ids)].copy()

print(f"Total clients: {len(all_client_ids)}")
print(f"Training clients: {len(train_client_ids)} (rows: {len(train_df)})")
print(f"Testing clients: {len(test_client_ids)} (rows: {len(test_df)})")

# Prepare features and labels for training
X_train = train_df[new_features_depth2].replace([np.inf, -np.inf], np.nan).fillna(0)
y_train = train_df['is_declining_label'].values

X_test = test_df[new_features_depth2].replace([np.inf, -np.inf], np.nan).fillna(0)
y_test = test_df['is_declining_label'].values

print("\nData split into training and testing sets based on client ID.")

Total clients: 32
Training clients: 25 (rows: 26581)
Testing clients: 7 (rows: 3419)

Data split into training and testing sets based on client ID.


Now that we have our client-holdout `train_df` and `test_df`, we can proceed to retrain our decision tree on `train_df` and evaluate its performance on `test_df`. We will use the same features (`new_features_depth2`) and `max_depth=2` as in the previous experiment for a direct comparison.

In [16]:
# Retrain the Decision Tree on the training data
tree_holdout = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree_holdout.fit(X_train, y_train)

# Evaluate the hand rule on the test data
hr_k20_holdout = precision_at_k(test_df["hand_rule_score"], y_test, 20)
hr_k50_holdout = precision_at_k(test_df["hand_rule_score"], y_test, 50)

# Evaluate the tree on the test data
tree_score_holdout = tree_holdout.predict_proba(X_test)[:, 1]
tr_k20_holdout = precision_at_k(tree_score_holdout, y_test, 20)
tr_k50_holdout = precision_at_k(tree_score_holdout, y_test, 50)

print("--- Client-Holdout Validation Results (max_depth=2, new features) ---")
print(f"Precision@20:  hand rule {hr_k20_holdout:.3f}   vs   tree {tr_k20_holdout:.3f}")
print(f"Precision@50:  hand rule {hr_k50_holdout:.3f}   vs   tree {tr_k50_holdout:.3f}")

print('\nTree structure (trained on holdout, max_depth=2):')
print(export_text(tree_holdout, feature_names=new_features_depth2))

--- Client-Holdout Validation Results (max_depth=2, new features) ---
Precision@20:  hand rule 0.650   vs   tree 0.450
Precision@50:  hand rule 0.620   vs   tree 0.520

Tree structure (trained on holdout, max_depth=2):
|--- avg_position <= 0.55
|   |--- word_count <= 687.00
|   |   |--- class: 0
|   |--- word_count >  687.00
|   |   |--- class: 0
|--- avg_position >  0.55
|   |--- content_age_days <= 287.50
|   |   |--- class: 1
|   |--- content_age_days >  287.50
|   |   |--- class: 0



After re-running the comparison with client-holdout validation:

* Both the hand rule and the decision tree showed a significant drop in performance compared to their in-sample scores.

* Crucially, the hand rule now outperformed the decision tree on the holdout data:

    * Hand Rule: Precision@20: 0.650, Precision@50: 0.620
    * Decision Tree (max_depth=2, new features): Precision@20: 0.450, Precision@50: 0.520
    
This indicates that the decision tree, when scored in-sample, likely overfit the training data, leading to an artificially inflated performance. The client-holdout validation reveals a more accurate picture of its generalization capability, where the simpler hand rule proved more robust.

### Save your work
**Colab:** *File → Save a copy in GitHub* (your submission) and *File → Save a copy in Drive*.

You now have the two core reflexes of applied ML: **discover before you model**, and **prefer a model you can read and can't fool**. That's the whole foundation the capstone builds on.